# Notebook 03 — Classical ML Pipeline
## HSV Histogram + LBP → XGBoost ReID → SORT Tracker

### Pipeline Overview
```
Detection crop
    ├── HSV histogram  (colour appearance)
    └── LBP histogram  (texture appearance)
            ↓
     Concatenate → L1 normalise
            ↓
     StandardScaler (fit on train only)
            ↓
     XGBoost binary classifier
     P(same person | feature_diff)
            ↓
     SORT tracker + IoU + appearance score
```

### Mathematical Foundations

#### 1. HSV Histogram
Convert BGR → HSV colour space. Build a 3D histogram with bins $(H=16, S=16, V=8)$.
Total feature length: $16 \times 16 \times 8 = 2048$ dimensions.
L1 normalise: $\hat{h}_i = h_i / \sum_j h_j$

#### 2. LBP (Local Binary Pattern)
For each pixel $p$ at position $(x,y)$ with $P$ neighbours at radius $R$:
$$\text{LBP}_{P,R}(x,y) = \sum_{p=0}^{P-1} s(g_p - g_c) \cdot 2^p$$
where $g_c$ = centre pixel value, $g_p$ = neighbour pixel value,
$s(x) = 1$ if $x \geq 0$, else $0$.
Using **uniform** LBP (at most 2 transitions in circular bit pattern):
histogram over $P+2 = 10$ uniform patterns. L1 normalise.

#### 3. Feature Fusion
$$\mathbf{f} = [\hat{h}_{\text{HSV}} \;\|\; \hat{h}_{\text{LBP}}]$$
Total: $2048 + 10 = 2058$ dimensions.

#### 4. XGBoost Pair Feature
Given two crops with features $\mathbf{f}_1$ and $\mathbf{f}_2$, the pair feature vector is:
$$\mathbf{x}_{\text{pair}} = [|\mathbf{f}_1 - \mathbf{f}_2|, \; d_{\cos}(\mathbf{f}_1, \mathbf{f}_2), \; d_{\chi^2}(\mathbf{f}_1, \mathbf{f}_2)]$$
where:
- $|\mathbf{f}_1 - \mathbf{f}_2|$: element-wise absolute difference
- $d_{\cos} = 1 - \frac{\mathbf{f}_1 \cdot \mathbf{f}_2}{\|\mathbf{f}_1\| \|\mathbf{f}_2\|}$: cosine distance
- $d_{\chi^2} = \sum_i \frac{(f_{1i} - f_{2i})^2}{f_{1i} + f_{2i} + \epsilon}$: chi-squared distance

#### 5. SORT Tracker + Appearance
Combined association score:
$$S(i, j) = w_{\text{IoU}} \cdot \text{IoU}(i,j) + w_{\text{app}} \cdot P_{\text{XGB}}(i,j)$$
where $w_{\text{IoU}} = w_{\text{app}} = 0.5$.

#### 6. Exponential Moving Average (EMA) Template Update
After a successful match, update the track template:
$$T_t = \alpha \cdot T_{t-1} + (1 - \alpha) \cdot \mathbf{f}_t$$
with $\alpha = 0.9$ (slow update — remembers history).

#### 7. Kalman Filter (SORT)
State vector: $\mathbf{x} = [u, v, s, r, \dot{u}, \dot{v}, \dot{s}]^T$
where $(u,v)$ = centre, $s$ = area, $r$ = aspect ratio.
Prediction: $\mathbf{x}_{k|k-1} = F \mathbf{x}_{k-1|k-1}$
Update: $\mathbf{x}_{k|k} = \mathbf{x}_{k|k-1} + K(z_k - H \mathbf{x}_{k|k-1})$

In [ ]:
import os, json, pickle, cv2, random
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from skimage.feature import local_binary_pattern
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
import xgboost as xgb
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
import warnings; warnings.filterwarnings('ignore')

# Load config
with open(os.path.join(
    r'D:\MTech\Sem-2\IT585-Advanced_ML\Project\AML-project',
    'visdrone_outputs', 'config.json')) as f:
    cfg = json.load(f)

TRAIN_DIR     = cfg['TRAIN_DIR']
VAL_DIR       = cfg['VAL_DIR']
OUTPUT_DIR    = cfg['OUTPUT_DIR']
VALID_CLASSES = set(cfg['VALID_CLASSES'])
VAL_SEQS      = cfg['VAL_SEQS']
random.seed(42); np.random.seed(42)
print('Classical ML Pipeline — Ready!')

## Part 1 — Feature Extraction

In [ ]:
# ──────────────────────────────────────────────────────────────────
# HSV HISTOGRAM FEATURE EXTRACTOR
# Converts BGR image crop to HSV colour space and computes a 3D
# histogram with bins (H=16, S=16, V=8), then L1-normalises.
# Total feature length: 16 × 16 × 8 = 2048 dimensions
# ──────────────────────────────────────────────────────────────────
def extract_hsv_histogram(bgr_crop, bins=(16, 16, 8)):
    """
    Extract L1-normalised HSV colour histogram from a BGR crop.

    Math:
        HSV[H] ∈ [0, 180], HSV[S] ∈ [0, 255], HSV[V] ∈ [0, 255]
        hist = 3D histogram with shape (bins[0], bins[1], bins[2])
        feature = hist.flatten() / sum(hist.flatten())

    Args:
        bgr_crop : numpy array (H, W, 3) in BGR format (from cv2)
        bins     : (H_bins, S_bins, V_bins) — number of histogram bins

    Returns:
        numpy array of shape (H_bins * S_bins * V_bins,) = (2048,)
    """
    hsv = cv2.cvtColor(bgr_crop, cv2.COLOR_BGR2HSV)

    # cv2.calcHist expects [image], [channels], mask, [bins], [ranges]
    hist = cv2.calcHist(
        [hsv],                      # image list
        [0, 1, 2],                  # all 3 channels
        None,                       # no mask
        list(bins),                 # bins per channel
        [0, 180, 0, 256, 0, 256]    # ranges: H=[0,180], S=[0,256], V=[0,256]
    )
    hist = hist.flatten().astype(np.float32)

    # L1 normalisation: divide by sum so values sum to 1
    total = hist.sum()
    if total > 0:
        hist /= total
    return hist


# ──────────────────────────────────────────────────────────────────
# LBP (LOCAL BINARY PATTERN) FEATURE EXTRACTOR
# Captures texture information — robust to illumination changes.
# Uses uniform LBP: patterns with at most 2 transitions in
# circular bit sequence. P=8 neighbours, R=1 radius.
# Histogram over P+2 = 10 uniform patterns. L1 normalised.
# ──────────────────────────────────────────────────────────────────
def extract_lbp_histogram(bgr_crop, points=8, radius=1):
    """
    Extract L1-normalised uniform LBP histogram from a BGR crop.

    Math:
        LBP(x,y) = Σ_{p=0}^{P-1} s(g_p - g_c) * 2^p
        where g_c = centre pixel, g_p = p-th neighbour
        s(x) = 1 if x >= 0 else 0

        Uniform LBP: at most 2 bit transitions in circular pattern
        → P+2 = 10 possible uniform codes for P=8
        All non-uniform patterns → single 'non-uniform' bin

    Args:
        bgr_crop : numpy array (H, W, 3) in BGR format
        points   : number of circular neighbours P (default 8)
        radius   : radius of circle R (default 1)

    Returns:
        numpy array of shape (points + 2,) = (10,)
    """
    gray = cv2.cvtColor(bgr_crop, cv2.COLOR_BGR2GRAY)
    lbp  = local_binary_pattern(gray, points, radius, method='uniform')

    # Histogram: bins = 0 to points+1 (uniform) + 1 (non-uniform)
    n_bins = points + 2
    hist, _ = np.histogram(lbp, bins=n_bins, range=(0, n_bins))
    hist = hist.astype(np.float32)

    # L1 normalisation
    total = hist.sum()
    if total > 0:
        hist /= total
    return hist


# ──────────────────────────────────────────────────────────────────
# FUSED FEATURE EXTRACTOR
# Concatenates HSV and LBP histograms into one feature vector.
# Total: 2048 + 10 = 2058 dimensions
# ──────────────────────────────────────────────────────────────────
def extract_classical_feature(bgr_crop, hsv_bins=(16,16,8), lbp_p=8, lbp_r=1):
    """
    Compute fused HSV + LBP feature for one crop.

    Math:
        f = [hsv_hist || lbp_hist]
        where || denotes concatenation

    Returns:
        numpy array of shape (2058,)
    """
    hsv_feat = extract_hsv_histogram(bgr_crop, bins=hsv_bins)
    lbp_feat = extract_lbp_histogram(bgr_crop, points=lbp_p, radius=lbp_r)
    return np.concatenate([hsv_feat, lbp_feat])


# Quick test
dummy = np.random.randint(0, 255, (128, 64, 3), dtype=np.uint8)
feat  = extract_classical_feature(dummy)
print(f'Feature vector length: {len(feat)}')
print(f'  HSV part: {16*16*8} dims')
print(f'  LBP part: {8+2} dims')
print(f'  Total   : {len(feat)} dims')
print(f'Feature sum (should be ~2.0 — two normalised parts): {feat.sum():.4f}')

## Part 2 — Extract Features for All Training Crops

In [ ]:
def read_visdrone_annotation(anno_path, valid_classes=None):
    cols = ['frame_id', 'target_id', 'x', 'y', 'w', 'h',
            'score', 'class_id', 'truncation', 'occlusion']
    df = pd.read_csv(anno_path, header=None, names=cols)
    df = df[df['score'] == 1]
    df = df[df['target_id'] > 0]
    if valid_classes:
        df = df[df['class_id'].isin(valid_classes)]
    return df[(df['w'] > 0) & (df['h'] > 0)].reset_index(drop=True)


def extract_features_for_sequence(split_dir, seq_name, valid_classes,
                                    crop_size=(64, 128), min_area=400):
    """
    Extract classical features for every valid detection in a sequence.

    Returns:
        features  : numpy array (N, 2058)
        metadata  : list of dicts with seq, frame_id, target_id, class_id, bbox
    """
    anno_path = os.path.join(split_dir, 'annotations', seq_name + '.txt')
    seq_dir   = os.path.join(split_dir, 'sequences', seq_name)
    df = read_visdrone_annotation(anno_path, valid_classes)

    features, metadata = [], []
    for fid, frame_df in df.groupby('frame_id'):
        img_path = os.path.join(seq_dir, f'{fid:07d}.jpg')
        if not os.path.exists(img_path):
            continue
        img = cv2.imread(img_path)
        if img is None:
            continue
        H_img, W_img = img.shape[:2]

        for _, row in frame_df.iterrows():
            x1 = max(0, int(row['x']))
            y1 = max(0, int(row['y']))
            x2 = min(W_img, int(row['x'] + row['w']))
            y2 = min(H_img, int(row['y'] + row['h']))
            if (x2-x1)*(y2-y1) < min_area:
                continue
            crop = cv2.resize(img[y1:y2, x1:x2], crop_size)
            feat = extract_classical_feature(crop)
            features.append(feat)
            metadata.append({
                'seq': seq_name, 'frame_id': int(row['frame_id']),
                'target_id': int(row['target_id']),
                'class_id': int(row['class_id']),
                'x': x1, 'y': y1, 'w': x2-x1, 'h': y2-y1
            })
    return np.array(features, dtype=np.float32), metadata


# Load split files
splits_dir = os.path.join(OUTPUT_DIR, 'splits')
with open(os.path.join(splits_dir, 'reid_train_sequences.txt')) as f:
    reid_train_seqs = f.read().splitlines()

print(f'Extracting classical features for {len(reid_train_seqs)} sequences...')
print('(Approx 10-20 mins depending on dataset size)')

all_features, all_metadata = [], []
for seq in tqdm(reid_train_seqs, desc='Extracting features'):
    feats, meta = extract_features_for_sequence(TRAIN_DIR, seq, VALID_CLASSES)
    if len(feats) > 0:
        all_features.append(feats)
        all_metadata.extend(meta)

all_features = np.vstack(all_features)
print(f'\n✅ Feature matrix shape: {all_features.shape}')
print(f'Total detections with features: {len(all_metadata):,}')

# Save features
np.save(os.path.join(OUTPUT_DIR, 'features', 'classical_features.npy'), all_features)
with open(os.path.join(OUTPUT_DIR, 'features', 'classical_metadata.pkl'), 'wb') as f:
    pickle.dump(all_metadata, f)
print('✅ Features saved!')

## Part 3 — Build XGBoost Training Data

In [ ]:
# ──────────────────────────────────────────────────────────────────
# PAIR FEATURE CONSTRUCTION FOR XGBOOST
# Given two feature vectors f1 and f2, construct a single vector
# representing their relationship:
#   |f1 - f2|       : absolute element-wise difference
#   cos_dist        : 1 - cosine_similarity
#   chi2_dist       : chi-squared distance
# This gives the XGBoost model information about BOTH
# local (per-dimension) and global (distributional) differences.
# ──────────────────────────────────────────────────────────────────
def compute_pair_feature(f1, f2, eps=1e-8):
    """
    Compute pair feature vector from two appearance feature vectors.

    Math:
        abs_diff = |f1 - f2|   (element-wise)
        cos_dist = 1 - (f1·f2) / (||f1|| * ||f2||)
        chi2     = Σ_i (f1_i - f2_i)² / (f1_i + f2_i + ε)

    Args:
        f1, f2 : feature vectors of same length
        eps    : small constant to avoid division by zero in chi2

    Returns:
        numpy array of shape (len(f1) + 2,) = (2060,)
    """
    abs_diff = np.abs(f1 - f2)

    # Cosine distance: 1 - cosine similarity
    norm1 = np.linalg.norm(f1) + eps
    norm2 = np.linalg.norm(f2) + eps
    cos_dist = np.array([1.0 - np.dot(f1, f2) / (norm1 * norm2)])

    # Chi-squared distance: captures histogram bin differences
    chi2 = np.array([np.sum((f1 - f2)**2 / (f1 + f2 + eps))])

    return np.concatenate([abs_diff, cos_dist, chi2])


def build_xgb_dataset(all_features, all_metadata, pos_per_id=5,
                       neg_ratio=2, seed=42):
    """
    Build (X, y) training data for XGBoost from feature matrix.

    Constructs positive pairs (same ID, diff frames)
    and negative pairs (diff ID, same sequence).
    """
    rng = random.Random(seed)

    # Group indices by (seq, target_id) for positive pairs
    id_to_idx   = defaultdict(list)
    seq_to_idx  = defaultdict(list)
    for i, m in enumerate(all_metadata):
        id_to_idx[(m['seq'], m['target_id'])].append(i)
        seq_to_idx[m['seq']].append(i)

    X_list, y_list = [], []

    # ── Positive pairs ────────────────────────────────────────────
    pos_count = 0
    for (seq, tid), indices in id_to_idx.items():
        if len(indices) < 2: continue
        for _ in range(pos_per_id):
            a, b = rng.sample(indices, 2)
            if all_metadata[a]['frame_id'] != all_metadata[b]['frame_id']:
                pf = compute_pair_feature(all_features[a], all_features[b])
                X_list.append(pf); y_list.append(1)
                pos_count += 1

    # ── Negative pairs ────────────────────────────────────────────
    n_neg = pos_count * neg_ratio
    neg_count = 0
    all_seq_names = list(seq_to_idx.keys())
    attempts = 0
    while neg_count < n_neg and attempts < n_neg * 10:
        seq  = rng.choice(all_seq_names)
        idxs = seq_to_idx[seq]
        if len(idxs) < 2: attempts += 1; continue
        a, b = rng.sample(idxs, 2)
        if all_metadata[a]['target_id'] != all_metadata[b]['target_id']:
            pf = compute_pair_feature(all_features[a], all_features[b])
            X_list.append(pf); y_list.append(0)
            neg_count += 1
        attempts += 1

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int32)
    print(f'Positive pairs: {pos_count:,}')
    print(f'Negative pairs: {neg_count:,}')
    print(f'X shape: {X.shape}, y shape: {y.shape}')
    return X, y


print('Building XGBoost training pairs...')
X_train, y_train = build_xgb_dataset(all_features, all_metadata)

## Part 4 — Train XGBoost ReID Classifier

In [ ]:
# ──────────────────────────────────────────────────────────────────
# XGBOOST CLASSIFIER TRAINING
# XGBoost is a gradient-boosted decision tree ensemble.
# It learns to predict P(same_identity | pair_feature_vector).
#
# Objective: binary:logistic
# Loss: log-loss = -[y·log(p) + (1-y)·log(1-p)]
# Output: probability ∈ [0, 1] that the pair belongs to same identity
# ──────────────────────────────────────────────────────────────────

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Train/val split for classifier evaluation
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.2,
    random_state=42, stratify=y_train
)
print(f'Classifier train: {len(X_tr):,} | val: {len(X_val):,}')

# XGBoost model with parameters from the pipeline spec
xgb_model = xgb.XGBClassifier(
    n_estimators=300,          # Number of boosting rounds
    max_depth=6,               # Max tree depth
    learning_rate=0.05,        # Step size shrinkage
    subsample=0.8,             # Fraction of samples per tree
    colsample_bytree=0.8,      # Fraction of features per tree
    eval_metric='logloss',     # Evaluation metric
    use_label_encoder=False,
    random_state=42,
    device='cuda' if __import__('torch').cuda.is_available() else 'cpu'
)

print('Training XGBoost classifier...')
xgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=50
)

# Evaluate
y_pred = xgb_model.predict(X_val)
y_prob = xgb_model.predict_proba(X_val)[:, 1]
print('\n=== XGBoost Classifier Evaluation ===')
print(classification_report(y_val, y_pred,
      target_names=['different ID', 'same ID']))
print(f'ROC-AUC: {roc_auc_score(y_val, y_prob):.4f}')

# Save model and scaler
xgb_model.save_model(os.path.join(OUTPUT_DIR, 'models', 'xgb_reid.json'))
with open(os.path.join(OUTPUT_DIR, 'models', 'classical_scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)
print('\n✅ XGBoost model and scaler saved!')

## Part 5 — SORT Tracker with Appearance (Classical)

In [ ]:
# ──────────────────────────────────────────────────────────────────
# KALMAN FILTER FOR SORT TRACKER
# State vector: [u, v, s, r, u_dot, v_dot, s_dot]
#   u, v   = centre x, y
#   s      = scale (area = w*h)
#   r      = aspect ratio (w/h, assumed constant)
#   u_dot, v_dot, s_dot = velocities
#
# Transition matrix F (constant velocity model):
#   u_{t+1} = u_t + u_dot
#   s_{t+1} = s_t + s_dot
# ──────────────────────────────────────────────────────────────────
class KalmanBoxTracker:
    """
    Kalman filter based tracker for a single bounding box.
    State: [cx, cy, area, aspect_ratio, cx_vel, cy_vel, area_vel]
    Observation: [cx, cy, area, aspect_ratio]
    """
    count = 0

    def __init__(self, bbox, feature=None):
        """
        Initialise tracker with first bounding box.
        bbox: [x1, y1, x2, y2]
        feature: appearance feature vector
        """
        from scipy.linalg import block_diag  # for covariance initialisation

        # State transition matrix F (7x7)
        self.F = np.eye(7)
        self.F[0, 4] = 1  # u += u_dot
        self.F[1, 5] = 1  # v += v_dot
        self.F[2, 6] = 1  # s += s_dot

        # Observation matrix H (4x7) — observe [u, v, s, r]
        self.H = np.zeros((4, 7))
        self.H[:4, :4] = np.eye(4)

        # Process noise covariance Q (uncertainty in dynamics)
        self.Q = np.eye(7)
        self.Q[4:, 4:] *= 0.01  # Low velocity noise

        # Measurement noise covariance R
        self.R = np.eye(4)
        self.R[2:, 2:] *= 10.0  # Higher uncertainty in scale/ratio

        # State covariance P
        self.P = np.eye(7) * 10
        self.P[4:, 4:] *= 1000  # High initial velocity uncertainty

        # State vector x (7x1)
        x1, y1, x2, y2 = bbox
        cx, cy = (x1+x2)/2, (y1+y2)/2
        w, h   = x2-x1, y2-y1
        s      = w * h          # area
        r      = w / (h + 1e-8) # aspect ratio
        self.x = np.array([[cx], [cy], [s], [r],
                            [0.], [0.], [0.]])  # zero initial velocity

        # Track management
        KalmanBoxTracker.count += 1
        self.id          = KalmanBoxTracker.count
        self.age         = 0      # frames since creation
        self.hits        = 0      # consecutive matches
        self.hit_streak  = 0
        self.time_since_update = 0

        # EMA appearance template — updated on each match
        self.template = feature.copy() if feature is not None else None

    def predict(self):
        """
        Predict next state using Kalman prediction step.
        x_{k|k-1} = F * x_{k-1|k-1}
        P_{k|k-1} = F * P_{k-1|k-1} * F^T + Q
        """
        # Prevent negative area
        if self.x[2] + self.x[6] <= 0:
            self.x[6] = 0

        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        self.age += 1
        self.time_since_update += 1
        return self.get_bbox()

    def update(self, bbox, feature=None, ema_alpha=0.9):
        """
        Update state with new measurement using Kalman update step.
        Kalman gain: K = P * H^T * (H * P * H^T + R)^{-1}
        State update: x = x + K * (z - H * x)
        Covariance:   P = (I - K * H) * P

        Also updates EMA appearance template:
            T_t = α * T_{t-1} + (1-α) * f_t
        """
        x1, y1, x2, y2 = bbox
        cx, cy = (x1+x2)/2, (y1+y2)/2
        w, h   = x2-x1, y2-y1
        s      = w * h
        r      = w / (h + 1e-8)
        z = np.array([[cx], [cy], [s], [r]])

        # Innovation covariance: S = H*P*H^T + R
        S = self.H @ self.P @ self.H.T + self.R
        # Kalman gain: K = P*H^T * S^{-1}
        K = self.P @ self.H.T @ np.linalg.inv(S)
        # State update
        self.x = self.x + K @ (z - self.H @ self.x)
        # Covariance update (Joseph form for numerical stability)
        I_KH = np.eye(7) - K @ self.H
        self.P = I_KH @ self.P

        self.time_since_update = 0
        self.hits += 1
        self.hit_streak += 1

        # EMA template update
        if feature is not None and self.template is not None:
            self.template = ema_alpha * self.template + (1-ema_alpha) * feature
        elif feature is not None:
            self.template = feature.copy()

    def get_bbox(self):
        """Convert state [cx, cy, s, r] back to [x1, y1, x2, y2]."""
        cx, cy, s, r = self.x[0,0], self.x[1,0], self.x[2,0], self.x[3,0]
        w = np.sqrt(max(s * r, 0))
        h = s / (w + 1e-8)
        return [cx - w/2, cy - h/2, cx + w/2, cy + h/2]


print('KalmanBoxTracker defined!')

In [ ]:
# ──────────────────────────────────────────────────────────────────
# SORT TRACKER WITH APPEARANCE
# Combines Kalman motion prediction with XGBoost appearance matching.
#
# Association score:
#   S(i,j) = w_iou * IoU(predicted_i, detection_j)
#           + w_app * P_xgb(template_i, feature_j)
#
# Hungarian algorithm finds the optimal assignment that
# maximises total association score.
# ──────────────────────────────────────────────────────────────────
def compute_iou_matrix(bboxes_a, bboxes_b):
    """
    Compute IoU between all pairs of bounding boxes.

    IoU(A, B) = |A ∩ B| / |A ∪ B|
    
    Args:
        bboxes_a: (N, 4) array of [x1, y1, x2, y2]
        bboxes_b: (M, 4) array of [x1, y1, x2, y2]

    Returns:
        iou_matrix: (N, M) array of IoU values
    """
    N, M = len(bboxes_a), len(bboxes_b)
    iou  = np.zeros((N, M))
    for i, a in enumerate(bboxes_a):
        for j, b in enumerate(bboxes_b):
            # Intersection
            xi1 = max(a[0], b[0]); yi1 = max(a[1], b[1])
            xi2 = min(a[2], b[2]); yi2 = min(a[3], b[3])
            inter = max(0, xi2-xi1) * max(0, yi2-yi1)
            # Union
            area_a = (a[2]-a[0]) * (a[3]-a[1])
            area_b = (b[2]-b[0]) * (b[3]-b[1])
            union  = area_a + area_b - inter + 1e-8
            iou[i, j] = inter / union
    return iou


class ClassicalSORT:
    """
    SORT tracker enhanced with XGBoost appearance ReID.

    Tracking loop per frame:
        1. Predict all track positions using Kalman
        2. Extract features from new detections
        3. Compute IoU matrix between predicted tracks and detections
        4. Compute appearance score matrix using XGBoost
        5. Combined score = w_iou * IoU + w_app * P_xgb
        6. Hungarian algorithm to find optimal assignment
        7. Update matched tracks, create new ones, delete old ones
    """

    def __init__(self, xgb_model, scaler, max_age=20, min_hits=3,
                  iou_threshold=0.3, w_iou=0.5, w_app=0.5, ema_alpha=0.9):
        self.xgb       = xgb_model
        self.scaler    = scaler
        self.max_age   = max_age        # Remove track if unseen for this many frames
        self.min_hits  = min_hits       # Only output track after this many matches
        self.iou_thr   = iou_threshold  # Minimum IoU to consider a match
        self.w_iou     = w_iou
        self.w_app     = w_app
        self.ema_alpha = ema_alpha
        self.trackers  = []
        self.frame_count = 0
        KalmanBoxTracker.count = 0  # Reset ID counter

    def update(self, detections, features):
        """
        Process one frame.

        Args:
            detections: (N, 4) array of [x1, y1, x2, y2]
            features  : (N, D) array of classical features

        Returns:
            List of [x1, y1, x2, y2, track_id] for active tracks
        """
        self.frame_count += 1

        # Step 1: Predict all tracks
        predicted_bboxes = [t.predict() for t in self.trackers]

        if len(detections) == 0:
            # Remove dead tracks
            self.trackers = [t for t in self.trackers
                             if t.time_since_update <= self.max_age]
            return []

        results = []
        if len(self.trackers) == 0:
            # No existing tracks — create new ones for all detections
            for i, det in enumerate(detections):
                feat = features[i] if features is not None else None
                self.trackers.append(KalmanBoxTracker(det, feat))
        else:
            # Step 2: Compute IoU matrix
            iou_matrix = compute_iou_matrix(predicted_bboxes, detections)

            # Step 3: Compute appearance score matrix
            app_matrix = np.zeros((len(self.trackers), len(detections)))
            if features is not None:
                for i, tracker in enumerate(self.trackers):
                    if tracker.template is None:
                        continue
                    pair_feats = []
                    for j in range(len(detections)):
                        pf = compute_pair_feature(tracker.template, features[j])
                        pair_feats.append(pf)
                    if pair_feats:
                        pf_array = np.array(pair_feats, dtype=np.float32)
                        pf_scaled = self.scaler.transform(pf_array)
                        probs = self.xgb.predict_proba(pf_scaled)[:, 1]
                        app_matrix[i] = probs

            # Step 4: Combined score S = w_iou * IoU + w_app * P_xgb
            score_matrix = self.w_iou * iou_matrix + self.w_app * app_matrix

            # Step 5: Hungarian assignment (maximise score = minimise -score)
            row_ind, col_ind = linear_sum_assignment(-score_matrix)

            matched, unmatched_dets = [], list(range(len(detections)))
            for r, c in zip(row_ind, col_ind):
                # Only accept match if combined score is above threshold
                if score_matrix[r, c] >= self.iou_thr:
                    matched.append((r, c))
                    if c in unmatched_dets:
                        unmatched_dets.remove(c)

            # Step 6: Update matched trackers
            matched_tracker_ids = set()
            for r, c in matched:
                feat = features[c] if features is not None else None
                self.trackers[r].update(detections[c], feat, self.ema_alpha)
                matched_tracker_ids.add(r)

            # Step 7: Create new trackers for unmatched detections
            for c in unmatched_dets:
                feat = features[c] if features is not None else None
                self.trackers.append(KalmanBoxTracker(detections[c], feat))

            # Mark unmatched trackers
            for i, t in enumerate(self.trackers):
                if i not in matched_tracker_ids and i < len(predicted_bboxes):
                    t.hit_streak = 0

        # Remove old tracks
        self.trackers = [t for t in self.trackers
                         if t.time_since_update <= self.max_age]

        # Output active, confirmed tracks
        for t in self.trackers:
            if (t.time_since_update == 0 and
                    (t.hit_streak >= self.min_hits or
                     self.frame_count <= self.min_hits)):
                bbox = t.get_bbox()
                results.append(bbox + [t.id])

        return results


print('ClassicalSORT defined!')
print('\n✅ Ready for Notebook 04: Deep Learning Pipeline')

## Part 6 — Run Classical Tracker on Validation Sequences

In [ ]:
def run_classical_tracker_on_sequence(split_dir, seq_name, xgb_model, scaler,
                                       valid_classes, dropped_frames=None,
                                       save_dir=None):
    """
    Run the classical SORT+XGBoost tracker on one sequence.

    Args:
        split_dir     : TRAIN_DIR or VAL_DIR
        seq_name      : sequence name
        xgb_model     : trained XGBoost classifier
        scaler        : fitted StandardScaler
        valid_classes : set of class IDs to keep
        dropped_frames: set of frame IDs to skip (simulates missing frames)
        save_dir      : directory to save track output txt

    Returns:
        track_results: list of [frame_id, track_id, x1, y1, x2, y2]
    """
    anno_path = os.path.join(split_dir, 'annotations', seq_name + '.txt')
    seq_dir   = os.path.join(split_dir, 'sequences', seq_name)
    df = read_visdrone_annotation(anno_path, valid_classes)

    if dropped_frames is None:
        dropped_frames = set()

    tracker = ClassicalSORT(xgb_model, scaler)
    track_results = []
    all_frame_ids = sorted(df['frame_id'].unique())

    for fid in all_frame_ids:
        # Skip dropped frames — simulates detection failure
        if fid in dropped_frames:
            # Still need to call tracker with empty detections
            tracker.update(np.empty((0, 4)), None)
            continue

        img_path = os.path.join(seq_dir, f'{fid:07d}.jpg')
        if not os.path.exists(img_path):
            tracker.update(np.empty((0, 4)), None)
            continue

        img = cv2.imread(img_path)
        if img is None:
            tracker.update(np.empty((0, 4)), None)
            continue
        H_img, W_img = img.shape[:2]

        # Get detections for this frame
        frame_df = df[df['frame_id'] == fid]
        detections, features = [], []
        for _, row in frame_df.iterrows():
            x1 = max(0, int(row['x']))
            y1 = max(0, int(row['y']))
            x2 = min(W_img, int(row['x'] + row['w']))
            y2 = min(H_img, int(row['y'] + row['h']))
            if (x2-x1)*(y2-y1) < 400: continue
            detections.append([x1, y1, x2, y2])
            crop = cv2.resize(img[y1:y2, x1:x2], (64, 128))
            features.append(extract_classical_feature(crop))

        detections = np.array(detections, dtype=np.float32) if detections else np.empty((0,4))
        features   = np.array(features,   dtype=np.float32) if features   else None

        tracks = tracker.update(detections, features)
        for t in tracks:
            x1, y1, x2, y2, tid = t
            track_results.append([fid, int(tid), x1, y1, x2, y2])

    # Save results
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        drop_label = f'drop{len(dropped_frames)}' if dropped_frames else 'nodrop'
        out_path = os.path.join(save_dir, f'{seq_name}_classical_{drop_label}.txt')
        with open(out_path, 'w') as f:
            for row in track_results:
                f.write(','.join(map(str, row)) + '\n')

    return track_results


# Load trained model
xgb_loaded = xgb.XGBClassifier()
xgb_loaded.load_model(os.path.join(OUTPUT_DIR, 'models', 'xgb_reid.json'))
with open(os.path.join(OUTPUT_DIR, 'models', 'classical_scaler.pkl'), 'rb') as f:
    scaler_loaded = pickle.load(f)

# Run on first validation sequence
test_seq = VAL_SEQS[0]
save_dir = os.path.join(OUTPUT_DIR, 'tracks', 'classical')
print(f'Running classical tracker on: {test_seq}')
results = run_classical_tracker_on_sequence(
    VAL_DIR, test_seq, xgb_loaded, scaler_loaded,
    VALID_CLASSES, save_dir=save_dir
)
print(f'Total track detections: {len(results):,}')
unique_ids = len(set(r[1] for r in results))
print(f'Unique track IDs assigned: {unique_ids}')
print('\n✅ Classical pipeline complete!')
print('Next: Run Notebook 04 for the Deep Learning pipeline')